<a href="https://colab.research.google.com/github/zixian0821-zoe/financial-computing-workshop/blob/main/enet_huber_v2(no_output).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Elastic Net + Huber (ENet+H) Model

Implementation of the Elastic Net + Huber (ENet+H) model from Gu, Kelly & Xiu (2020) 'Empirical Asset Pricing via Machine Learning'.

This notebook implements:
- 920-dimensional feature construction (Paper Eq. 21): 94 characteristics × 9 macros (including constant) = 846 interactions + 74 SIC2 dummies
- Accelerated Proximal Gradient (APG) optimizer for Huber loss + Elastic Net regularization
- Expanding window training with 12-year rolling validation
- Hyperparameter tuning via validation set
- Out-of-sample R² computation

**Memory Note:** This notebook requires ~8-10 GB RAM. Use Colab Pro if free Colab runs out of memory.

In [1]:
# Cell 1: Setup - imports and Google Drive mount
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
drive.mount('/content/drive')
print('Google Drive mounted successfully.')

Mounted at /content/drive
Google Drive mounted successfully.


In [2]:
# Cell 2: Load parquet data with all columns needed
PARQUET_PATH = '/content/drive/MyDrive/industry_project/preprocess_data.parquet'

df = pd.read_parquet(PARQUET_PATH)
df['DATE'] = pd.to_datetime(df['DATE'])

print(f'Data shape: {df.shape}')
print(f'Date range: {df["DATE"].min().date()} to {df["DATE"].max().date()}')
print(f'Stocks:     {df["permno"].nunique():,}')
df.head(3)

Data shape: (3712808, 109)
Date range: 1957-04-30 to 2016-12-30
Stocks:     29,825


,permno,DATE,mvel1,beta,betasq,chmom,dolvol,idiovol,indmom,mom1m,...,exret,exret_lead1,tbl,d/p,e/p,b/m,tms,dfy,ntis,svar
0,10000,1986-02-28,-0.381006,0.0,0.0,0.0,0.000000,0.0,0.310144,0.000485,...,-0.262443,0.359385,0.0707,0.037492,0.068845,0.583517,0.0251,0.0139,-0.019172,0.001920
1,10000,1986-03-31,-0.505829,0.0,0.0,0.0,0.000000,0.0,0.358323,-0.952720,...,0.359385,-0.103792,0.0706,0.035167,0.064120,0.536377,0.0135,0.0144,-0.017914,0.001089
2,10000,1986-04-30,-0.410132,0.0,0.0,0.0,-0.539206,0.0,0.281704,0.932882,...,-0.103792,-0.227556,0.0656,0.033571,0.060779,0.519628,0.0110,0.0150,-0.016420,0.001374


In [3]:
# Cell 3: Identify 94 characteristic columns, 8 macros, create SIC2 dummies ONCE

# ── Non-characteristic columns ────────────────────────────────────────────────
# NOTE: parquet stores macro vars as 'b/m', 'd/p', 'e/p' (with slashes).
#       Use actual parquet column names here to avoid including them in CHAR_COLS_94
#       and to avoid incorrectly excluding the stock-level 'bm' characteristic.
NON_CHAR_COLS = {'permno', 'DATE', 'ret', 'rf', 'exret', 'exret_lead1', 'sic2',
                 'tbl', 'b/m', 'd/p', 'e/p', 'ntis', 'tms', 'dfy', 'svar'}
#                       ^^^   ^^^   ^^^  ← actual parquet column names (with slash)

CHAR_COLS_94 = sorted([c for c in df.columns if c not in NON_CHAR_COLS])
MACRO_COLS   = ['tbl', 'b/m', 'd/p', 'e/p', 'ntis', 'tms', 'dfy', 'svar']
#                      ^^^   ^^^   ^^^  ← must match parquet names exactly

print(f'94 stock characteristics: {len(CHAR_COLS_94)}')   # should be 94
print(f' 8 macro variables:       {MACRO_COLS}')

# ── Create SIC2 dummies ONCE for the full dataset ─────────────────────────────
# Use all SIC2 codes that appear in the training period (up to 2016-12)
sic2_dummies = pd.get_dummies(df['sic2'].astype(str), prefix='sic2', drop_first=False)
SIC2_DUMMY_COLS = sorted(sic2_dummies.columns.tolist())
print(f'SIC2 dummy columns:       {len(SIC2_DUMMY_COLS)}')

# Attach dummies to the main dataframe (once — avoids per-year recreation)
df = pd.concat([df, sic2_dummies], axis=1)
del sic2_dummies
import gc; gc.collect()

print(f'\ndf shape after adding SIC2 dummies: {df.shape}')
print(f'Total features will be: {len(CHAR_COLS_94)} * 9 + {len(SIC2_DUMMY_COLS)} = {len(CHAR_COLS_94)*9 + len(SIC2_DUMMY_COLS)}')
# Expected: 94 * 9 + 74 = 920  ✓

94 stock characteristics: 94
 8 macro variables:       ['tbl', 'b/m', 'd/p', 'e/p', 'ntis', 'tms', 'dfy', 'svar']
SIC2 dummy columns:       74

df shape after adding SIC2 dummies: (3712808, 183)
Total features will be: 94 * 9 + 74 = 920


## Section 1: Model Configuration

In [4]:
# Cell 5: Model configuration and constants

TARGET           = 'exret_lead1'
VALIDATION_END   = pd.Timestamp('1986-12-31')
TEST_END         = pd.Timestamp('2016-12-31')
VALIDATION_YEARS = 12

# ── Hyperparameter grid (Table A.5) ───────────────────────────────────────────
LAM_GRID = np.logspace(-4, -1, 20)   # lambda in (1e-4, 1e-1), 20 grid points
RHO      = 0.5                        # elastic-net mixing, FIXED per Table A.5

# ── Clean: drop rows with NaN in required columns ────────────────────────────
REQUIRED = CHAR_COLS_94 + MACRO_COLS + [TARGET, 'mvel1']
df_clean = df.dropna(subset=REQUIRED).copy()
df_clean = df_clean.sort_values(['DATE', 'permno']).reset_index(drop=True)

print(f'Clean rows:  {len(df_clean):,} / {len(df):,}')

# ── Test years (same as OLS-3: 1987–2016) ────────────────────────────────────
test_years = sorted(
    df_clean.loc[
        (df_clean['DATE'] > VALIDATION_END) &
        (df_clean['DATE'] <= TEST_END),
        'DATE'
    ].dt.year.unique()
)
print(f'Test period: {test_years[0]}–{test_years[-1]}  ({len(test_years)} years)')
print(f'Lambda grid: {len(LAM_GRID)} values in [{LAM_GRID[0]:.0e}, {LAM_GRID[-1]:.0e}]')
print(f'Rho (fixed): {RHO}')

Clean rows:  3,712,808 / 3,712,808
Test period: 1987–2016  (30 years)
Lambda grid: 20 values in [1e-04, 1e-01]
Rho (fixed): 0.5


## Section 2: Feature Construction and Helper Functions

In [5]:
# Cell 7: Core functions for ENet+H model

def build_features_920(data, char_cols, macro_cols, sic2_dummies_cols):
    """
    Build 920-dimensional feature vector per paper Eq. 21:
    z_{i,t} = x_t ⊗ c_{i,t} (846 interactions) + 74 SIC2 dummies

    x_t = [1, tbl, b/m, d/p, e/p, ntis, tms, dfy, svar] (9-dim: constant + 8 macros)
    c_{i,t} = 94 characteristics

    Interactions: each char × each macro_with_constant = 94 × 9 = 846
    """
    chars = data[char_cols].values.astype(np.float32)  # (N, 94)

    # x_t: constant=1 + 8 macros = 9 columns
    macro_with_const = np.column_stack([
        np.ones(len(data), dtype=np.float32),
        data[macro_cols].values.astype(np.float32)
    ])  # (N, 9)

    # Kronecker product: (N, 94, 1) * (N, 1, 9) -> (N, 94, 9) -> reshape to (N, 846)
    interactions = (chars[:, :, None] * macro_with_const[:, None, :]).reshape(len(data), -1)

    # SIC2 dummies (already created as 0/1 columns)
    sic2_vals = data[sic2_dummies_cols].values.astype(np.float32)  # (N, 74)

    return np.hstack([interactions, sic2_vals])  # (N, 920)


def huber_loss(residuals, xi):
    """
    Huber loss (Paper Eq. 6):
    H(r; xi) = r^2                if |r| <= xi
             = 2*xi*|r| - xi^2   otherwise
    """
    abs_r = np.abs(residuals)
    return np.where(abs_r <= xi, residuals**2, 2*xi*abs_r - xi**2)


def huber_grad(X, y, theta, xi):
    """
    Gradient of (1/N) sum H(y - X@theta, xi) w.r.t. theta
    """
    N = len(y)
    residuals = y - X @ theta
    mask = np.abs(residuals) <= xi
    dH = np.where(mask, 2*residuals, 2*xi*np.sign(residuals))
    return -(1.0/N) * (X.T @ dH)


def prox_enet(theta, gamma, lam, rho):
    """
    Proximal operator for elastic net:
    prox(theta) = sign(theta) * max(|theta| - gamma*lambda*(1-rho), 0) / (1 + gamma*lambda*rho)
    """
    tau = gamma * lam * (1 - rho)
    shrunk = np.sign(theta) * np.maximum(np.abs(theta) - tau, 0)
    return shrunk / (1 + gamma * lam * rho)


def apg_huber_enet(X, y, lam, rho, xi, gamma=None, max_iter=2000, tol=1e-6,
                    theta_init=None, intercept=True):
    """
    Accelerated Proximal Gradient for:
    min_theta (1/N) sum H(y - X*theta, xi) + lambda*(1-rho)*sum|theta_j| + (1/2)*lambda*rho*sum(theta_j^2)

    Parameters
    ----------
    X : (N, P) feature matrix (should be standardized)
    y : (N,) target (should be centered if intercept=True)
    lam : regularization strength lambda
    rho : elastic net mixing (0=Lasso, 0.5=ENet, 1=Ridge)
    xi : Huber threshold
    gamma : step size (if None, computed from data)
    max_iter : maximum iterations
    tol : convergence tolerance
    theta_init : warm start initialization
    intercept : if True, y should already be centered

    Returns
    -------
    theta : (P,) coefficient vector
    """
    N, P = X.shape

    if gamma is None:
        # Step size: 1/L where L is Lipschitz constant of Huber gradient
        # L <= 2/N * max_eigenvalue(X'X)
        # Efficient approximation using power iteration (few iterations)
        v = np.random.randn(P).astype(np.float32)
        for _ in range(20):
            v = X.T @ (X @ v)
            norm_v = np.linalg.norm(v) + 1e-12
            v = v / norm_v
        max_eig = np.linalg.norm(X.T @ (X @ v))
        L = 2.0 * max_eig / N
        gamma = 1.0 / (L + 1e-12)

    theta = theta_init.copy() if theta_init is not None else np.zeros(P, dtype=np.float64)
    theta_old = theta.copy()

    for m in range(max_iter):
        # Step 1: Gradient of Huber loss
        grad = huber_grad(X, y, theta, xi)

        # Step 2: Gradient step
        theta_tilde = theta - gamma * grad

        # Step 3: Proximal step for elastic net
        theta_bar = prox_enet(theta_tilde, gamma, lam, rho)

        # Step 4: Nesterov momentum
        theta_new = theta_bar + (m / (m + 3)) * (theta_bar - theta_old)

        # Convergence check
        diff = np.linalg.norm(theta_new - theta)
        if diff < tol * (1 + np.linalg.norm(theta)):
            break

        theta_old = theta.copy()
        theta = theta_new

    return theta


def standardize(X, mean=None, std=None):
    """
    Standardize features to mean=0, std=1.
    If mean and std are provided, use them (for test data).
    Otherwise compute from X (for train data).

    Returns: X_std, mean, std
    """
    if mean is None:
        mean = X.mean(axis=0)
    if std is None:
        std = X.std(axis=0)

    std = np.where(std < 1e-8, 1.0, std)  # Avoid division by zero
    X_std = (X - mean) / std
    return X_std, mean, std


def oos_r2(y_true, y_pred, benchmark_var=None):
    """
    Out-of-sample R^2 (Eq. 19 in paper):
    R^2_oos = 1 - SS_pred / SS_zero_benchmark

    where SS_pred = sum((y_true - y_pred)^2)
    and SS_zero_benchmark = sum(y_true^2) if using zero benchmark
    """
    ss_res = np.sum((y_true - y_pred)**2)
    if benchmark_var is None:
        ss_tot = np.sum(y_true**2)
    else:
        ss_tot = len(y_true) * benchmark_var

    if ss_tot == 0:
        return np.nan
    return 1.0 - (ss_res / ss_tot)

print('All core functions defined successfully.')

All core functions defined successfully.


## Section 3: Expanding Window ENet+H Estimation (1987-2016)

Per **Appendix D**: expanding train + rolling 12-year validation. Parameters estimated from **training data alone**.

For each test year $y$:
1. **Train** (expanding): 1957 to $(y-13)$
2. **Val** (rolling 12y): $(y-12)$ to $(y-1)$ --- used ONLY to tune $\lambda$ and $\xi$
3. Standardize features using train statistics; center $y$ using train mean
4. Grid search $\lambda$: for each $\lambda$, APG on train $\to$ evaluate val Huber loss
5. Refit APG on **train only** with best $\lambda^*$
6. Predict test year

**Note:** With 920 features and $\sim$1M training rows, each test year takes several minutes. Total runtime: $\sim$2--4 hours.

In [6]:
# Cell 9: Main estimation loop — expanding window ENet+H

import time, gc

all_preds = []           # accumulate predictions for pooled R²
year_info  = []          # per-year diagnostics

for year in test_years:

    t0 = time.time()

    # ── 1. DATE-based split (identical scheme to OLS-3) ──────────────────────
    train_cutoff = pd.Timestamp(f'{year - 1}-12-31')
    val_start    = pd.Timestamp(f'{year - 1 - VALIDATION_YEARS}-12-31')

    df_train = df_clean[df_clean['DATE'] <= val_start]
    df_val   = df_clean[(df_clean['DATE'] > val_start) & (df_clean['DATE'] <= train_cutoff)]
    df_test  = df_clean[df_clean['DATE'].dt.year == year]

    if min(len(df_train), len(df_val), len(df_test)) == 0:
        continue

    # ── 2. Build 920-dim features ────────────────────────────────────────────
    X_train = build_features_920(df_train, CHAR_COLS_94, MACRO_COLS, SIC2_DUMMY_COLS)
    X_val   = build_features_920(df_val,   CHAR_COLS_94, MACRO_COLS, SIC2_DUMMY_COLS)
    X_test  = build_features_920(df_test,  CHAR_COLS_94, MACRO_COLS, SIC2_DUMMY_COLS)

    y_train = df_train[TARGET].to_numpy(dtype=np.float64)
    y_val   = df_val[TARGET].to_numpy(dtype=np.float64)
    y_test  = df_test[TARGET].to_numpy(dtype=np.float64)

    # ── 3. Standardize features (train stats only); center y ─────────────────
    X_train_s, X_mean, X_std = standardize(X_train)
    X_val_s,   _,      _     = standardize(X_val,  X_mean, X_std)
    X_test_s,  _,      _     = standardize(X_test, X_mean, X_std)

    y_mu    = float(y_train.mean())
    y_tr_c  = y_train - y_mu          # centered training target
    y_val_c = y_val   - y_mu          # centered validation target

    # ── 4. Compute Huber threshold xi ────────────────────────────────────────
    #   Initial ridge (fast closed-form) on train → val residuals → xi
    P = X_train_s.shape[1]
    try:
        XtX = (X_train_s.T @ X_train_s).astype(np.float64)
        Xty = (X_train_s.T @ y_tr_c).astype(np.float64)
        theta_ridge_init = np.linalg.solve(
            XtX / len(y_train) + 0.01 * np.eye(P),
            Xty / len(y_train)
        )
    except np.linalg.LinAlgError:
        theta_ridge_init = np.zeros(P)

    val_resid = y_val_c - X_val_s @ theta_ridge_init
    xi = float(np.percentile(np.abs(val_resid), 99.9))
    xi = max(xi, 1e-4)

    # ── 5. Grid search over lambda on validation ─────────────────────────────
    best_val_loss = np.inf
    best_lam      = LAM_GRID[0]
    best_theta    = None
    theta_warm    = None

    for lam in LAM_GRID:
        theta = apg_huber_enet(
            X_train_s, y_tr_c, lam, RHO, xi,
            max_iter=1500, tol=1e-5,
            theta_init=theta_warm
        )
        theta_warm = theta.copy()          # warm start for next lambda

        # Validation Huber loss (without penalty)
        v_loss = float(np.mean(huber_loss(y_val_c - X_val_s @ theta, xi)))
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_lam      = lam
            best_theta    = theta.copy()

    # ── 6. Predict test year (un-center) ─────────────────────────────────────
    y_pred = (X_test_s @ best_theta) + y_mu

    # ── 7. Store predictions ─────────────────────────────────────────────────
    res = df_test[['DATE', 'permno', 'mvel1']].copy().reset_index(drop=True)
    res['y_true'] = y_test
    res['y_pred'] = y_pred
    all_preds.append(res)

    n_nonzero = int(np.sum(np.abs(best_theta) > 1e-8))
    elapsed   = time.time() - t0

    year_info.append({
        'year': year, 'n_train': len(df_train), 'n_val': len(df_val),
        'n_test': len(df_test), 'lam': best_lam, 'xi': xi,
        'nnz': n_nonzero, 'sec': elapsed
    })

    print(
        f'Year {year} | train {len(df_train):>9,} | val {len(df_val):>9,} | '
        f'test {len(df_test):>7,} | lam={best_lam:.2e} xi={xi:.4f} '
        f'nnz={n_nonzero:>4d}/{P} | {elapsed:.0f}s'
    )

    # Free large arrays
    del X_train, X_val, X_test, X_train_s, X_val_s, X_test_s, XtX, Xty
    gc.collect()

print('\nEstimation complete.')

Year 1987 | train   472,278 | val   764,497 | test  82,404 | lam=5.46e-03 xi=1.8988 nnz=  41/920 | 24297s


KeyboardInterrupt: 

## Section 4: Results Summary

In [ ]:
# Cell 11: Pooled R² — All / Top 1000 / Bottom 1000  (same format as OLS-3 notebook)

results = pd.concat(all_preds, ignore_index=True)
print(f'Total OOS predictions: {len(results):,}')
print(f'Test period:           {results["DATE"].min().date()} to {results["DATE"].max().date()}')

# ── All stocks ───────────────────────────────────────────────────────────────
r2_all = oos_r2(results['y_true'].values, results['y_pred'].values)

# ── Top 1000: 1000 largest by mvel1 each month ───────────────────────────────
top1000 = (
    results
    .sort_values(['DATE', 'mvel1'], ascending=[True, False])
    .groupby('DATE', sort=False).head(1000)
)
r2_top = oos_r2(top1000['y_true'].values, top1000['y_pred'].values)

# ── Bottom 1000: 1000 smallest by mvel1 each month ──────────────────────────
bot1000 = (
    results
    .sort_values(['DATE', 'mvel1'], ascending=[True, True])
    .groupby('DATE', sort=False).head(1000)
)
r2_bot = oos_r2(bot1000['y_true'].values, bot1000['y_pred'].values)

# ── Table 1 style output ─────────────────────────────────────────────────────
print()
print('=' * 72)
print(f"  {'Subsample':<35} {'OOS R^2':>10}  {'Paper ENet+H':>15}")
print('-' * 72)
print(f"  {'All stocks (panel)':<35} {r2_all*100:>+10.4f}%  {'~ +0.11%':>15}")
print(f"  {'Top 1,000 (largest mvel1)':<35} {r2_top*100:>+10.4f}%  {'~ +0.25%':>15}")
print(f"  {'Bottom 1,000 (smallest mvel1)':<35} {r2_bot*100:>+10.4f}%  {'~ +0.34%':>15}")
print('=' * 72)

# ── Per-year diagnostics ─────────────────────────────────────────────────────
info_df = pd.DataFrame(year_info)
print(f'\nPer-year: lambda median={info_df["lam"].median():.2e}, '
      f'xi median={info_df["xi"].median():.4f}, '
      f'non-zero coefs median={info_df["nnz"].median():.0f}/{P}')

## Section 5: Yearly R^2 Time Series

In [ ]:
# Cell 13: Yearly R² time series plot + lambda path

results['year'] = results['DATE'].dt.year
top1000['year'] = top1000['DATE'].dt.year
bot1000['year'] = bot1000['DATE'].dt.year

yearly_r2 = []
for yr in sorted(results['year'].unique()):
    m_all = results['year'] == yr
    m_top = top1000['year'] == yr
    m_bot = bot1000['year'] == yr
    yearly_r2.append({
        'year': yr,
        'All':      oos_r2(results.loc[m_all,'y_true'], results.loc[m_all,'y_pred']) * 100,
        'Top 1000': oos_r2(top1000.loc[m_top,'y_true'], top1000.loc[m_top,'y_pred']) * 100,
        'Bot 1000': oos_r2(bot1000.loc[m_bot,'y_true'], bot1000.loc[m_bot,'y_pred']) * 100,
    })

df_yr = pd.DataFrame(yearly_r2).set_index('year')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: yearly R²
ax = axes[0]
df_yr.plot(ax=ax, marker='o', markersize=3, linewidth=1.2)
ax.axhline(0, color='k', linewidth=0.7, linestyle='--')
ax.set_ylabel(r'$R^2_{oos}$ (%)')
ax.set_title('ENet+H: Yearly OOS $R^2$')
ax.legend(); ax.grid(alpha=0.3)

# Right: best lambda per year (Figure 3 style)
ax2 = axes[1]
idf = pd.DataFrame(year_info)
ax2.semilogy(idf['year'], idf['lam'], 'o-', markersize=4, color='tab:orange')
ax2.set_ylabel(r'Best $\lambda$')
ax2.set_title('Tuned $\\lambda$ per test year')
ax2.grid(alpha=0.3)

# Second y-axis: number of non-zero coefficients
ax3 = ax2.twinx()
ax3.plot(idf['year'], idf['nnz'], 's--', markersize=3, color='tab:blue', alpha=0.6)
ax3.set_ylabel('# non-zero coefs', color='tab:blue')

plt.tight_layout(); plt.show()

## Section 6: Notes and Implementation Details

In [ ]:
# Cell 15: Summary statistics and notes

print('IMPLEMENTATION NOTES:')
print('='*80)
print()
print('Model: Elastic Net + Huber (ENet+H) from Gu, Kelly & Xiu (2020)')
print()
print('Feature Construction (Paper Eq. 21):')
print('  - 94 rank-normalized stock characteristics')
print('  - 8 macro variables + constant = 9-dim macro vector')
print('  - Kronecker product: 94 chars × 9 macros = 846 interactions')
print('  - 74 SIC2 industry dummies')
print('  - Total: 846 + 74 = 920 features')
print()
print('Loss Function and Regularization:')
print('  - Huber loss (Paper Eq. 6) for robustness to outliers')
print('  - Elastic Net penalty: lambda * [(1-rho) * L1 + (1/2) * rho * L2]')
print('  - Rho = 0.5 (fixed) per Table A.5')
print()
print('Optimization: Accelerated Proximal Gradient (APG, Algorithm 1, Appendix B.1)')
print('  - Nesterov momentum acceleration')
print('  - Convergence tolerance: 1e-6')
print('  - Max iterations: 2000')
print()
print('Hyperparameter Tuning:')
print('  - Lambda: 20 log-spaced values in [1e-4, 1e-1]')
print('  - Grid search on validation set using Huber loss')
print('  - Warm starting along lambda path for efficiency')
print()
print('Sample Splitting (Appendix D):')
print('  - Train: 1957 to (test_year - 13), expanding window')
print('  - Validation: (test_year - 12) to (test_year - 1), rolling 12 years')
print('  - Test: test_year (one year at a time)')
print('  - Parameters estimated from training data alone')
print('  - SIC2 dummies created from training data; unknown codes = 0')
print()
print('Data Preprocessing:')
print('  - Features standardized (mean=0, std=1) using training statistics only')
print('  - Target centered using training mean')
print('  - Predictions un-centered before evaluation')
print('  - Huber threshold xi: 99.9th percentile of |validation residuals|')
print()
print('Evaluation Metric:')
print('  - Out-of-sample R^2 (Eq. 19): 1 - SS_residuals / SS_zero_benchmark')
print('  - Zero benchmark: zero prediction')
print('  - Reported for: All stocks, Top 1000, Bottom 1000 by market value')
print()
print('Memory and Computational Notes:')
print('  - Features stored as float32 to reduce memory usage')
print('  - Requires ~8-10 GB RAM (use Colab Pro if needed)')
print('  - Power iteration (20 iterations) for Lipschitz constant estimation')
print('  - Warm starting reduces overall computation time')
print()
print('='*80)